#### Inisialisasi

In [31]:
import findspark
findspark.init()  # Menghubungkan VS Code ke Apache Spark lokal

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
from pyspark.ml.feature import (
    StringIndexer, VectorAssembler,
    MinMaxScaler, OneHotEncoder
)

spark = SparkSession.builder \
    .appName("CellTower_Preprocessing_Notebook") \
    .config("spark.sql.shuffle.partitions", "200") \
    .getOrCreate()
print("Sesi Spark Berhasil Dibuat")

Sesi Spark Berhasil Dibuat


#### Definisi Schema & Load Data

In [32]:
schema = StructType([
    StructField("index",         LongType(),    True),
    StructField("radio",         StringType(),  True),
    StructField("MCC",           IntegerType(), True),
    StructField("MNC",           IntegerType(), True),
    StructField("TAC",           IntegerType(), True),
    StructField("CID",           LongType(),    True),
    StructField("unit",          IntegerType(), True),
    StructField("LON",           FloatType(),   True),
    StructField("LAT",           FloatType(),   True),
    StructField("RANGE",         IntegerType(), True),
    StructField("SAM",           IntegerType(), True),
    StructField("changeable",    IntegerType(), True),
    StructField("created",       LongType(),    True),
    StructField("updated",       LongType(),    True),
    StructField("averageSignal", IntegerType(), True),
    StructField("Country",       StringType(),  True),
    StructField("Network",       StringType(),  True),
    StructField("Continent",     StringType(),  True),
])

HDFS_CSV_INPUT = "hdfs://localhost:9000/Project_akhir/Asia towers.csv"
df_raw = (spark.read
          .format("csv")
          .option("header", "true")          
          .option("mode", "DROPMALFORMED")   
          .schema(schema)                    
          .load(HDFS_CSV_INPUT))

print(f"Total data mentah dari CSV berhasil di-load: {df_raw.count():,}")
df_raw.printSchema()

Total data mentah dari CSV berhasil di-load: 13,375,989
root
 |-- index: long (nullable = true)
 |-- radio: string (nullable = true)
 |-- MCC: integer (nullable = true)
 |-- MNC: integer (nullable = true)
 |-- TAC: integer (nullable = true)
 |-- CID: long (nullable = true)
 |-- unit: integer (nullable = true)
 |-- LON: float (nullable = true)
 |-- LAT: float (nullable = true)
 |-- RANGE: integer (nullable = true)
 |-- SAM: integer (nullable = true)
 |-- changeable: integer (nullable = true)
 |-- created: long (nullable = true)
 |-- updated: long (nullable = true)
 |-- averageSignal: integer (nullable = true)
 |-- Country: string (nullable = true)
 |-- Network: string (nullable = true)
 |-- Continent: string (nullable = true)



#### Pemilihan data asean

In [33]:
SE_ASIA = [
    "Brunei", "Cambodia", "East Timor", "Indonesia", "Laos",
    "Malaysia", "Myanmar", "Philippines", "Singapore", "Thailand", "Vietnam"
]

df = df_raw.filter(F.col("Country").isin(SE_ASIA))
df = df.drop("Continent", "averageSignal", "changeable")

print(f"Jumlah baris setelah filter ASEAN: {df.count():,}")

Jumlah baris setelah filter ASEAN: 4,122,331


#### Cleaning

In [34]:
# Hapus baris dengan nilai null pada kolom krusial
KOLOM_KRUSIAL = [
    "radio", "MCC", "MNC", "TAC", "LON", "LAT",
    "RANGE", "SAM", "Country", "Network", "created", "updated"
]

df_no_null = df.dropna(subset=KOLOM_KRUSIAL)
print(f"Jumlah baris setelah hapus null: {df_no_null.count():,}")

Jumlah baris setelah hapus null: 4,122,331


In [35]:
# filter nilai nol
df_no_zero = df_no_null.filter(
    (F.col("MCC")   != 0) &
    (F.col("MNC")   != 0) &
    (F.col("TAC")   != 0) &
    (F.col("SAM")   != 0) &
    (F.col("RANGE") != 0)
)
print(f"Jumlah baris setelah filter nol: {df_no_zero.count():,}")

Jumlah baris setelah filter nol: 4,042,284


In [36]:
# Validasi Format String (Country & Network)
## Memastikan nama negara dan jaringan hanya mengandung karakter yang valid.

df_valid_str = df_no_zero.filter(
    F.col("Country").rlike(r"^[a-zA-Z\s]{2,}$") &
    F.col("Network").rlike(r"^[a-zA-Z0-9\s\-\.\&\+\(\)\/]{2,}$")
)
print(f"Jumlah baris setelah validasi string: {df_valid_str.count():,}")

Jumlah baris setelah validasi string: 3,857,904


In [37]:
# Validasi Presisi Koordinat
## Koordinat LON dan LAT harus memiliki minimal 2 angka di belakang koma untuk akurasi spasial.

df_valid_coord = df_valid_str.filter(
    (F.length(F.regexp_extract(F.abs(F.col("LON")).cast("string"), r"\.(\d+)", 1)) >= 2) &
    (F.length(F.regexp_extract(F.abs(F.col("LAT")).cast("string"), r"\.(\d+)", 1)) >= 2)
)
print(f"Jumlah baris setelah validasi koordinat: {df_valid_coord.count():,}")

Jumlah baris setelah validasi koordinat: 3,857,066


In [38]:
# Hitung Usia Data & Filter Teknis
## Hitung `data_age_days` dari timestamp `updated`
## Filter batas geografis ASEAN (LON 90–145, LAT -11–28)
## Pastikan `updated >= created` dan jenis radio valid

current_timestamp = 1715360400  # Unix timestamp referensi (Mei 2024)

df_clean = df_valid_coord.withColumn(
    "data_age_days",
    (F.lit(current_timestamp) - F.col("updated")) / (3600 * 24)
)

# Clamp negatif ke 0
df_clean = df_clean.withColumn(
    "data_age_days",
    F.when(F.col("data_age_days") < 0, 0).otherwise(F.col("data_age_days"))
)

# Filter validitas teknis & batas area ASEAN
df_clean = df_clean.filter(
    (F.col("LON").between(90, 145)) &
    (F.col("LAT").between(-11, 28)) &
    (F.col("updated") >= F.col("created")) &
    (F.col("radio").isin(["GSM", "UMTS", "LTE", "NR", "CDMA"]))
)

print(f"Jumlah baris setelah filter teknis: {df_clean.count():,}")

Jumlah baris setelah filter teknis: 3,857,065


#### Feature Engineering & Labeling Data
Membuat fitur baru yang lebih informatif:
- created_year : tahun pertama kali tower dicatat
- data_age : selisih waktu antara created dan updated
- ever_updated : flag apakah tower pernah diperbarui
- generasi : kategori generasi jaringan (2G/3G/4G/5G)
- jangkauan : kategori area (Urban/Suburban/Rural)
- keandalan_data : tingkat keandalan berdasarkan SAM
- LAT_VIS, LON_VIS : koordinat dibulatkan untuk visualisasi

In [39]:
df_enc = df_clean.withColumn("created_year", F.year(F.from_unixtime("created"))) \
                 .withColumn("updated_year", F.year(F.from_unixtime("updated"))) \
                 .withColumn("LON_VIS", F.col("LON")) \
                 .withColumn("LAT_VIS", F.col("LAT"))


current_timestamp = 1715360400
df_enc = df_enc.withColumn("data_age_days", 
    F.when((F.lit(current_timestamp) - F.col("updated")) / 86400 < 0, 0)
    .otherwise((F.lit(current_timestamp) - F.col("updated")) / 86400)
)

# Labeling
indexer_radio = StringIndexer(inputCol="radio", outputCol="radio_index", handleInvalid="skip")
df_enc = indexer_radio.fit(df_enc).transform(df_enc)

indexer_country = StringIndexer(inputCol="Country", outputCol="country_index", handleInvalid="skip")
df_enc = indexer_country.fit(df_enc).transform(df_enc)

df_enc = df_enc.withColumn("generasi",
    F.when(F.col("radio") == "GSM",  "2G")
     .when(F.col("radio") == "UMTS", "3G")
     .when(F.col("radio") == "LTE",  "4G")
     .when(F.col("radio") == "NR",   "5G")
     .when(F.col("radio") == "CDMA",   "2G")
)

indexer_gen = StringIndexer(inputCol="generasi", outputCol="generasi_index", handleInvalid="skip")
df_enc = indexer_gen.fit(df_enc).transform(df_enc)

df_enc = df_enc.withColumn("jangkauan",
    F.when(F.col("RANGE") < 1000,  "Urban")
     .when(F.col("RANGE") <= 5000, "Suburban")
     .otherwise("Rural")
)

indexer_jangkauan = StringIndexer(inputCol="jangkauan", outputCol="jangkauan_index", handleInvalid="skip")
df_enc = indexer_jangkauan.fit(df_enc).transform(df_enc)

df_enc = df_enc.withColumn("keandalan_data",
    F.when(F.col("SAM") >= 10, "high")
     .when(F.col("SAM") >= 3,  "medium")
     .otherwise("low")
)

# lihat sedikit hasil pelabelan data kategorikal
df_enc.select("radio", "radio_index", "Country", "country_index",
              "generasi", "jangkauan", "keandalan_data").show(10)

+-----+-----------+---------+-------------+--------+---------+--------------+
|radio|radio_index|  Country|country_index|generasi|jangkauan|keandalan_data|
+-----+-----------+---------+-------------+--------+---------+--------------+
|  GSM|        2.0|Singapore|          5.0|      2G| Suburban|        medium|
| UMTS|        0.0|Singapore|          5.0|      3G| Suburban|           low|
|  GSM|        2.0|Singapore|          5.0|      2G| Suburban|        medium|
|  GSM|        2.0|Singapore|          5.0|      2G| Suburban|          high|
|  GSM|        2.0|Singapore|          5.0|      2G| Suburban|           low|
|  LTE|        1.0|Singapore|          5.0|      4G| Suburban|        medium|
|  GSM|        2.0|Singapore|          5.0|      2G|    Rural|        medium|
| UMTS|        0.0|Singapore|          5.0|      3G|    Rural|          high|
| UMTS|        0.0|Singapore|          5.0|      3G| Suburban|          high|
| UMTS|        0.0|Singapore|          5.0|      3G| Suburban|  

#### Normalisasi Data

In [40]:
# Normalisasi Fitur Spasial (LAT, LON, RANGE)
assembler_spatial = VectorAssembler(inputCols=["LON", "LAT", "RANGE"], outputCol="spatial_raw")
df_enc = assembler_spatial.transform(df_enc)
scaler_spatial = MinMaxScaler(inputCol="spatial_raw", outputCol="features_spatial")
df_enc = scaler_spatial.fit(df_enc).transform(df_enc)

# Normalisasi Identitas Jaringan (MCC, MNC, Unit)
assembler_mcc = VectorAssembler(inputCols=["MCC", "MNC", "unit"], outputCol="mcc_mnc_raw")
df_enc = assembler_mcc.transform(df_enc)
scaler_mcc = MinMaxScaler(inputCol="mcc_mnc_raw", outputCol="mcc_mnc_scaled")
df_enc = scaler_mcc.fit(df_enc).transform(df_enc)

# Penyatuan Fitur Utama untuk K-Means & Random Forest
assembler_pred = VectorAssembler(inputCols=["features_spatial"], outputCol="prediction_features")
df_enc = assembler_pred.transform(df_enc)

# Normalisasi Indikator Keandalan (Untuk GMM)
assembler_rel = VectorAssembler(inputCols=["SAM", "data_age_days"], outputCol="reliability_raw")
df_enc = assembler_rel.transform(df_enc)
scaler_rel = MinMaxScaler(inputCol="reliability_raw", outputCol="reliability_metrics")
df_enc = scaler_rel.fit(df_enc).transform(df_enc)

# Tampilkan hasil normalisasi (Data berubah menjadi array ter-scaling antara 0-1)
df_enc.select("spatial_raw", "features_spatial", "reliability_raw", "reliability_metrics").show(5, truncate=False)

+----------------------------------------------+--------------------------------------------------------------+--------------------------+------------------------------------------+
|spatial_raw                                   |features_spatial                                              |reliability_raw           |reliability_metrics                       |
+----------------------------------------------+--------------------------------------------------------------+--------------------------+------------------------------------------+
|[103.82789611816406,1.431656002998352,1000.0] |[0.23572463359108956,0.32377906582370936,5.330128926731749E-4]|[3.0,2638.6697106481483]  |[1.846125444223935E-5,0.1628370725528273] |
|[103.62579345703125,1.3094329833984375,1000.0]|[0.2315713721284642,0.3205900037733216,5.330128926731749E-4]  |[2.0,3991.8467939814814]  |[9.230627221119675E-6,0.24639198445722624]|
|[103.83888244628906,1.4252469539642334,1000.0]|[0.23595040544862753,0.32361183991879644,5

#### Penyimpanan

In [41]:
KOLOM_FINAL = [
    "index", "MCC", "MNC", "TAC", "CID", "unit", "radio", "radio_index", 
    "generasi", "generasi_index", "LON", "LAT", "LON_VIS", "LAT_VIS", "RANGE", "SAM", 
    "created", "updated", "created_year", "updated_year", "data_age_days", 
    "Country", "Network", "country_index", "jangkauan", "jangkauan_index",
    "features_spatial", "prediction_features", "reliability_metrics"
]

HDFS_OUTPUT = "hdfs://localhost:9000/Project_akhir/data_bersih_asean"

print("Menyimpan dataset hasil preprocessing ke HDFS...")
df_enc.select(KOLOM_FINAL).write.mode("overwrite").parquet(HDFS_OUTPUT)
print(f"Sukses! Data bersih disimpan dengan aman di HDFS: {HDFS_OUTPUT}")

Menyimpan dataset hasil preprocessing ke HDFS...
Sukses! Data bersih disimpan dengan aman di HDFS: hdfs://localhost:9000/Project_akhir/data_bersih_asean
